> **Note:** This notebook now loads IMDB via `datasets.load_dataset("imdb")` (previously it used `tf.keras.datasets.imdb`, a pre-tokenized/integer-encoded copy of the corpus, requiring TensorFlow just for data loading). Outputs below are cleared -- run all cells to reproduce metrics. The baseline metrics reported in the README (88.00% accuracy / 88.02% F1) were produced by running `python -m src.train_baseline` (the production script in `src/`, same data path and hyperparameters as this notebook), not by executing this notebook directly.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

# Note: this baseline used to load data via tf.keras.datasets.imdb (pre-tokenized,
# integer-encoded reviews) while the DistilBERT notebook used Hugging Face
# `datasets`. That meant two different, slightly inconsistent copies of the same
# corpus and an unnecessary TensorFlow dependency just for data loading. Both
# notebooks now load the same raw-text IMDB dataset via `datasets.load_dataset`,
# so there is one consistent data path for both models.

In [ ]:
imdb = load_dataset("imdb")

texts_train = imdb["train"]["text"]
y_train = np.array(imdb["train"]["label"])

texts_test = imdb["test"]["text"]
y_test = np.array(imdb["test"]["label"])

print("Train samples:", len(texts_train))
print("Test samples:", len(texts_test))
print("Label (0=neg, 1=pos):", y_train[0])

In [ ]:
# Quick look at a raw review (already plain text, no decoding needed)
print(texts_train[0][:500])

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,      # cap vocab size
    stop_words="english",    # remove common English stopwords
)

X_train = vectorizer.fit_transform(texts_train)
X_test  = vectorizer.transform(texts_test)

X_train.shape, X_test.shape


In [ ]:
clf = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)


In [ ]:
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred,
    average="binary",  # positive class = 1
    pos_label=1
)

print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Neg", "Pred Pos"],
            yticklabels=["True Neg", "True Pos"])
plt.title("IMDB Sentiment – Confusion Matrix")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()
